In [13]:
import polars as pl
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from aligner import GeneLoader

def get_idx(filename):
    return int(Path(filename).stem.split("_")[-1])


home = Path.cwd()

data_path = home / "popns"

files = [f for f in data_path.iterdir() if f.suffix == ".parquet"]
print(len(files))
file_path = files[20]

df = pl.read_parquet(file_path)
print(df.columns)
idx = get_idx(file_path)


49
['seq', 'states', 'conv_score', 'embedding', 'mut_array', 'll', 'live', 'update', 'fit_cos_dist', 'fit_hmm_prob']


In [3]:
L = GeneLoader("hg38_primary")

In [4]:
fig = go.Figure(go.Scatter(
    x=df["fit_hmm_prob"].to_numpy(),
    y=df["fit_cos_dist"].to_numpy(),
    mode="markers",
    marker=dict(size=5, opacity=0.7),
))

fig.update_layout(
    title="HMM Probability vs Cosine Distance",
    xaxis_title="fit_hmm_prob",
    yaxis_title="fit_cos_dist",
)
fig.show()

In [5]:
fig = go.Figure(go.Scatter(
    x=df["conv_score"].to_numpy(),
    y=df["fit_cos_dist"].to_numpy(),
    mode="markers",
    marker=dict(size=5, opacity=0.7),
))

fig.update_layout(
    title="Conservation Score vs Cosine Distance",
    xaxis_title="conv_score",
    yaxis_title="fit_cos_dist",
)
fig.show()

In [6]:
mut_array = np.array(df["mut_array"].to_list())

cos_dist = df["fit_cos_dist"].to_numpy()
sort_idx = np.argsort(cos_dist)
mut_array_sorted = mut_array[sort_idx]
cos_dist_sorted = cos_dist[sort_idx]

fig = make_subplots(rows=1, cols=2, column_widths=[0.85, 0.15], shared_yaxes=True)

fig.add_trace(go.Heatmap(
    z=mut_array_sorted,
    colorscale=[[0, "purple"], [1, "yellow"]],
    zmin=0,
    zmax=1,
    showscale=False,
), row=1, col=1)

# discrete legend via invisible scatter traces
for val, color, label in [(0, "purple", "No mutation"), (1, "yellow", "Mutation")]:
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=10, color=color, symbol="square"),
        name=label,
        showlegend=True,
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=cos_dist_sorted,
    y=np.arange(len(cos_dist_sorted)),
    mode="lines",
    line=dict(color="red", width=2),
    name="cos_dist",
), row=1, col=2)

fig.update_layout(
    title="Mutation Locations (sorted by cosine distance)",
    xaxis=dict(title="Position"),
    xaxis2=dict(title="Cosine Distance"),
    yaxis=dict(title="Individual (sorted by cos_dist)"),
)
fig.show()

In [7]:
entry = L.get_idx(idx)
model_conservation = 1 - mut_array.mean(axis=0)
phylop_scores = entry["conv"]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True)

fig.add_trace(go.Scatter(
    x=np.arange(len(model_conservation)),
    y=model_conservation,
    mode="lines",
    name="Model Conservation",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=np.arange(len(phylop_scores)),
    y=phylop_scores,
    mode="lines",
    name="PhyloP Score",
), row=2, col=1)

fig.update_layout(
    title="Model Conservation vs PhyloP",
    xaxis2=dict(title="Position"),
    yaxis=dict(title="Model Conservation"),
    yaxis2=dict(title="PhyloP Score"),
)
fig.show()

In [8]:
# get original state array from the loader
original_states = entry["states"]

# get mutated state arrays from the parquet
state_arrays = np.array(df["states"].to_list())

state_names = list(L.state_dict.keys())
n_states = len(state_names)

colors = px.colors.qualitative.Alphabet[n_states:]

colorscale = []
for i in range(n_states):
    colorscale.append([i / (n_states - 1), colors[i]])
    if i < n_states - 1:
        colorscale.append([(i + 1) / (n_states - 1) - 0.0001, colors[i]])

n_repeats = 5
all_states = np.vstack([
    state_arrays,
    np.full((1, len(original_states)), -1),
    np.tile(original_states, (n_repeats, 1)),
])

label_array = np.full(all_states.shape, "", dtype=object)
for i in range(n_states):
    label_array[all_states == i] = state_names[i]
label_array[all_states == -1] = "separator"

n_total = all_states.shape[0]
yticklabels = [""] * n_total
yticklabels[n_total - n_repeats // 2 - 1] = "original"

fig = go.Figure(go.Heatmap(
    z=all_states,
    text=label_array,
    hovertemplate="Position: %{x}<br>State: %{text}<extra></extra>",
    colorscale=colorscale,
    zmin=0,
    zmax=n_states - 1,
    showscale=True,
    colorbar=dict(
        title="State",
        tickvals=[i + 0.5 for i in range(n_states)],
        ticktext=state_names,
    ),
))

fig.update_layout(
    title="Nucleotide States: Original vs Mutated",
    xaxis_title="Position",
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(n_total)),
        ticktext=yticklabels,
    ),
)
fig.show()

In [9]:
cos_dist = df["fit_cos_dist"].to_numpy()
inv_weights = 1 / (cos_dist + 1e-10)
weights = inv_weights #/ inv_weights.sum()

n_positions = state_arrays.shape[1]
weighted_state_array = np.zeros(n_positions, dtype=int)
certainty = np.zeros(n_positions)

for pos in range(n_positions):
    state_counts = np.zeros(n_states)
    for i, state in enumerate(state_arrays[:, pos]):
        state_counts[state] += weights[i]
    weighted_state_array[pos] = np.argmax(state_counts)
    # certainty = mean cos_dist of individuals that agree on this state
    agreeing = state_arrays[:, pos] == weighted_state_array[pos]
    certainty[pos] = cos_dist[agreeing].mean()

# aggregate into runs of same state
positions, states, certs = [], [], []
i = 0
while i < n_positions:
    s = weighted_state_array[i]
    run = [i]
    while i + 1 < n_positions and weighted_state_array[i + 1] == s:
        i += 1
        run.append(i)
    positions.append(np.mean(run))
    states.append(s)
    certs.append(np.mean(certainty[run]))
    i += 1

fig = go.Figure()
for s in range(n_states):
    mask = [i for i, st in enumerate(states) if st == s]
    if mask:
        fig.add_trace(go.Scatter(
            x=[positions[i] for i in mask],
            y=[certs[i] for i in mask],
            mode="markers",
            marker=dict(color=colors[s], size=8),
            name=state_names[s],
        ))

fig.update_layout(
    title="Aggregated State Certainty by Position",
    xaxis_title="Position",
    yaxis_title="Certainty",
)
fig.show()

In [10]:
# build confusion matrix
state_names = list(L.state_dict.keys())
n_states = len(state_names)
confusion = np.zeros((n_states, n_states))
counts = np.zeros((n_states, n_states))

for f in files:
    idx = get_idx(f.name)
    entry = L.get_idx(idx)
    original_states_arr = np.array(entry["states"])
    
    df = pl.read_parquet(f)
    state_arrays = np.array(df["states"].to_list())
    
    for pos in range(len(original_states_arr)):
        orig = original_states_arr[pos]
        for row in state_arrays:
            counts[orig, row[pos]] += 1

confusion = counts / counts.sum(axis=1, keepdims=True)

fig = go.Figure(go.Heatmap(
    z=confusion,
    x=state_names,
    y=state_names,
    colorscale="Viridis",
    text=np.round(confusion, 2),
    hovertemplate="From: %{y}<br>To: %{x}<br>Fraction: %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title="State Confusion Matrix",
    xaxis_title="Mutated State",
    yaxis_title="Original State",
)
fig.show()

/tmp/ipykernel_72295/1874180364.py:20: RuntimeWarning: invalid value encountered in divide
  confusion = counts / counts.sum(axis=1, keepdims=True)


In [11]:
import itertools

bases = ["A", "T", "C", "G"]
codons = ["".join(c) for c in itertools.product(bases, repeat=3)]  # 64 codons
codon_to_int = {c: i for i, c in enumerate(codons)}

counts = np.zeros((n_states, 64))

for f in files:
    idx = get_idx(f.name)
    entry = L.get_idx(idx)
    original_states_arr = np.array(entry["states"])
    
    df2 = pl.read_parquet(f)
    seqs = df2["seq"].to_list()
    mut_arrays = np.array(df2["mut_array"].to_list())
    
    for pos in range(0, len(original_states_arr) - 2, 3):
        orig = original_states_arr[pos]
        for i, row in enumerate(mut_arrays):
            if row[pos] > 0 or row[pos+1] > 0 or row[pos+2] > 0:
                codon = "".join(seqs[i][pos:pos+3])
                if codon in codon_to_int:
                    counts[orig, codon_to_int[codon]] += 1

confusion = counts / counts.sum(axis=1, keepdims=True)

fig = go.Figure(go.Heatmap(
    z=confusion,
    x=codons,
    y=state_names,
    colorscale="Viridis",
    hovertemplate="From: %{y}<br>To: %{x}<br>Fraction: %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title="State to Codon Confusion Matrix",
    xaxis_title="Mutated Codon",
    yaxis_title="Original State",
)
fig.show()

/tmp/ipykernel_72295/906585815.py:26: RuntimeWarning: invalid value encountered in divide
  confusion = counts / counts.sum(axis=1, keepdims=True)


In [12]:
# compute marginal nucleotide frequencies per state from mutations
nuc_to_int = {"A": 0, "T": 1, "C": 2, "G": 3}
bases = ["A", "T", "C", "G"]
codons = ["".join(c) for c in itertools.product(bases, repeat=3)]
codon_to_int = {c: i for i, c in enumerate(codons)}

nuc_counts = np.zeros((n_states, 4))
codon_counts = np.zeros((n_states, 64))

for f in files:
    idx = get_idx(f.name)
    entry = L.get_idx(idx)
    original_states_arr = np.array(entry["states"])
    
    df2 = pl.read_parquet(f)
    seqs = df2["seq"].to_list()
    mut_arrays = np.array(df2["mut_array"].to_list())
    cos_dists = df2["fit_cos_dist"].to_numpy()
    weights = cos_dists / cos_dists.sum()  # normalize weights
    
    # single nucleotide counts
    for pos in range(len(original_states_arr)):
        orig = original_states_arr[pos]
        for i, row in enumerate(mut_arrays):
            if row[pos] > 0:
                nuc = nuc_to_int.get(seqs[i][pos], 0)
                nuc_counts[orig, nuc] += weights[i]
    
    # codon counts
    for pos in range(0, len(original_states_arr) - 2, 3):
        orig = original_states_arr[pos]
        for i, row in enumerate(mut_arrays):
            if row[pos] > 0 or row[pos+1] > 0 or row[pos+2] > 0:
                codon = "".join(seqs[i][pos:pos+3])
                if codon in codon_to_int:
                    codon_counts[orig, codon_to_int[codon]] += weights[i]

# compute marginal frequencies per state
nuc_freqs = nuc_counts / nuc_counts.sum(axis=1, keepdims=True)

# compute expected codon frequencies under independence
expected = np.zeros((n_states, 64))
for i, codon in enumerate(codons):
    for s in range(n_states):
        expected[s, i] = (nuc_freqs[s, nuc_to_int[codon[0]]] *
                          nuc_freqs[s, nuc_to_int[codon[1]]] *
                          nuc_freqs[s, nuc_to_int[codon[2]]])

# observed frequencies
observed = codon_counts / codon_counts.sum(axis=1, keepdims=True)

# ratio of observed to expected
ratio = np.where(expected > 0, observed / expected, 0)

fig = go.Figure(go.Heatmap(
    z=ratio,
    x=codons,
    y=state_names,
    colorscale="RdBu_r",
    zmid=1,
    hovertemplate="From: %{y}<br>Codon: %{x}<br>Obs/Exp: %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title="Codon Enrichment (Observed / Expected under independence)",
    xaxis_title="Codon",
    yaxis_title="Original State",
)
fig.show()

/tmp/ipykernel_72295/2095017420.py:39: RuntimeWarning: invalid value encountered in divide
  nuc_freqs = nuc_counts / nuc_counts.sum(axis=1, keepdims=True)
/tmp/ipykernel_72295/2095017420.py:50: RuntimeWarning: invalid value encountered in divide
  observed = codon_counts / codon_counts.sum(axis=1, keepdims=True)
